# Enrich USA Olympic Gold Medals with Event Dates (Multi-method Scraper)

This notebook:

1. Reads `olympic_medals.csv` and inspects field names/types.
2. Filters to USA (default: `country_3_letter_code == "USA"` and `medal_type == "GOLD"`).
3. Prepares a unique event list for scraping (deduping team medals so we don't overcount or overscrape).
4. Scrapes event date using fallbacks:
   - **Method 1:** Wikipedia page infobox `Dates` field (HTML)
   - **Method 2:** Wikipedia → **Wikidata** (`P585` / `P582` / `P580`)
   - **Method 3:** Wikipedia **wikitext** infobox parameter (`| dates =` / `| date =`) via MediaWiki API
5. Merges the recovered `event_date_end` back onto the medal rows and exports CSVs.

**Design choice:** Many Wikipedia event pages list a date *range* (e.g., `2–8 February 2022`). For event studies, we map the medal award date conservatively to the **end of the range** (`2022-02-08`).

Outputs:
- `usa_event_dates_lookup.csv` (one row per unique USA gold event)
- `usa_olympic_medals_with_event_dates.csv` (row-level medals + event dates)
- `usa_medal_daily_counts_and_score_ALL.csv` (deduped USA medal counts per day)
- `usa_medal_daily_counts_and_score_SUMMER.csv` (deduped USA summer medal counts per day)
- `usa_medal_daily_counts_and_score_WINTER.csv` (deduped USA winter medal counts per day)

In [48]:
%pip -q install pandas requests beautifulsoup4 lxml tqdm python-dateutil rapidfuzz

Note: you may need to restart the kernel to use updated packages.


In [49]:
import re
import time
from pathlib import Path
from datetime import datetime
from typing import Optional, Dict, Any, List

import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from dateutil import parser as dtparser
from rapidfuzz import fuzz

pd.set_option("display.max_columns", 200)

# ---- Paths ----
DATA_PATH = Path("../data/olympic_medals.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("/mnt/data/olympic_medals.csv")

CACHE_DIR = Path("cache_wiki")
CACHE_DIR.mkdir(exist_ok=True)

# ---- HTTP session ----
USER_AGENT = "QSS20-EventStudyBot/1.1 (educational; contact: your_email@example.com)"
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": USER_AGENT})

WIKI_API = "https://en.wikipedia.org/w/api.php"
WIKI_REST = "https://en.wikipedia.org/api/rest_v1/page/html"
WIKIDATA_API = "https://www.wikidata.org/w/api.php"


In [ ]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Columns:", list(df.columns))
display(df.head(5))
display(df.dtypes)

Shape: (21697, 12)
Columns: ['discipline_title', 'slug_game', 'event_title', 'event_gender', 'medal_type', 'participant_type', 'participant_title', 'athlete_url', 'athlete_full_name', 'country_name', 'country_code', 'country_3_letter_code']


,discipline_title,slug_game,event_title,event_gender,medal_type,participant_type,participant_title,athlete_url,athlete_full_name,country_name,country_code,country_3_letter_code
0,Curling,beijing-2022,Mixed Doubles,Mixed,GOLD,GameTeam,Italy,https://olympics.com/en/athletes/stefania-cons...,Stefania CONSTANTINI,Italy,IT,ITA
1,Curling,beijing-2022,Mixed Doubles,Mixed,GOLD,GameTeam,Italy,https://olympics.com/en/athletes/amos-mosaner,Amos MOSANER,Italy,IT,ITA
2,Curling,beijing-2022,Mixed Doubles,Mixed,SILVER,GameTeam,Norway,https://olympics.com/en/athletes/kristin-skaslien,Kristin SKASLIEN,Norway,NO,NOR
3,Curling,beijing-2022,Mixed Doubles,Mixed,SILVER,GameTeam,Norway,https://olympics.com/en/athletes/magnus-nedreg...,Magnus NEDREGOTTEN,Norway,NO,NOR
4,Curling,beijing-2022,Mixed Doubles,Mixed,BRONZE,GameTeam,Sweden,https://olympics.com/en/athletes/almida-de-val,Almida DE VAL,Sweden,SE,SWE


discipline_title         object
slug_game                object
event_title              object
event_gender             object
medal_type               object
participant_type         object
participant_title        object
athlete_url              object
athlete_full_name        object
country_name             object
country_code             object
country_3_letter_code    object
dtype: object

## 2) Filter to USA and 2000 - 2024 only

In [62]:
# ---- Filter controls ----
FILTER_COUNTRY_3 = "USA"
FILTER_MEDAL_TYPE = "ALL"   # "ALL" or one of: "GOLD", "SILVER", "BRONZE"
YEAR_MIN, YEAR_MAX = 2000, 2024

# ---- Basic hygiene ----
cols_clean = ["discipline_title","slug_game","event_title","event_gender","medal_type","country_3_letter_code"]
for c in cols_clean:
    df[c] = df[c].astype(str).str.strip()

# ---- Country + medal filter ----
df_us = df[df["country_3_letter_code"] == FILTER_COUNTRY_3].copy()
if FILTER_MEDAL_TYPE.upper() != "ALL":
    df_us = df_us[df_us["medal_type"] == FILTER_MEDAL_TYPE].copy()

# ---- Extract year + restrict to 2000–2024 (Summer + Winter) ----
df_us["year"] = pd.to_numeric(df_us["slug_game"].str.extract(r"(\d{4})")[0], errors="coerce")
df_us = df_us.dropna(subset=["year"]).copy()
df_us["year"] = df_us["year"].astype(int)

df_us = df_us[(df_us["year"] >= YEAR_MIN) & (df_us["year"] <= YEAR_MAX)].copy()

# ---- Season assignment (valid for 2000–2024: Summer years divisible by 4) ----
df_us["season"] = np.where(df_us["year"] % 4 == 0, "S", "W")

print(f"Filtered USA rows ({YEAR_MIN}–{YEAR_MAX}):", df_us.shape)
display(df_us.head(5))

# ---- Deduplicate medals at medal-award level (team events appear once per athlete) ----
MEDAL_KEY = ["slug_game","discipline_title","event_title","event_gender","medal_type","country_3_letter_code"]
df_us_dedup_medals = df_us.drop_duplicates(subset=MEDAL_KEY).copy()
print("Unique medal awards after dedupe:", df_us_dedup_medals.shape[0])

# ---- Build events to scrape ----
EVENT_KEY = ["slug_game","discipline_title","event_title","event_gender"]
events_us = (
    df_us_dedup_medals[EVENT_KEY]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"Unique USA events to scrape ({YEAR_MIN}–{YEAR_MAX}):", len(events_us))

display(events_us.head(10))

Filtered USA rows (2000–2024): (870, 14)


,discipline_title,slug_game,event_title,event_gender,medal_type,participant_type,participant_title,athlete_url,athlete_full_name,country_name,country_code,country_3_letter_code,year,season
16,Freestyle Skiing,beijing-2022,Men's Freeski Halfpipe,Men,SILVER,Athlete,NaN,https://olympics.com/en/athletes/david-wise,David WISE,United States of America,US,USA,2022,W
17,Freestyle Skiing,beijing-2022,Men's Freeski Halfpipe,Men,BRONZE,Athlete,NaN,https://olympics.com/en/athletes/alex-ferreira,Alex FERREIRA,United States of America,US,USA,2022,W
20,Freestyle Skiing,beijing-2022,Men's Freeski Big Air,Men,SILVER,Athlete,NaN,https://olympics.com/en/athletes/colby-stevenson,Colby STEVENSON,United States of America,US,USA,2022,W
27,Freestyle Skiing,beijing-2022,Women's Moguls,Women,SILVER,Athlete,NaN,https://olympics.com/en/athletes/kauf,Jaelin KAUF,United States of America,US,USA,2022,W
30,Freestyle Skiing,beijing-2022,Mixed Team Aerials,Mixed,GOLD,GameTeam,United States,NaN,NaN,United States of America,US,USA,2022,W


Unique medal awards after dedupe: 814
Unique USA events to scrape (2000–2024): 704


,slug_game,discipline_title,event_title,event_gender
0,beijing-2022,Freestyle Skiing,Men's Freeski Halfpipe,Men
1,beijing-2022,Freestyle Skiing,Men's Freeski Big Air,Men
2,beijing-2022,Freestyle Skiing,Women's Moguls,Women
3,beijing-2022,Freestyle Skiing,Mixed Team Aerials,Mixed
4,beijing-2022,Freestyle Skiing,Women's Aerials,Women
5,beijing-2022,Freestyle Skiing,Men's Freeski Slopestyle,Men
6,beijing-2022,Snowboard,Women's Snowboard Halfpipe,Women
7,beijing-2022,Snowboard,Mixed Team Snowboard Cross,Mixed
8,beijing-2022,Snowboard,Women's Snowboard Cross,Women
9,beijing-2022,Snowboard,Women's Snowboard Slopestyle,Women


## 3) Prepare unique events to scrape

We scrape **one date per unique event** (by games + discipline + event + gender), then merge back to all medals in that event.


In [63]:
EVENT_KEY = ["slug_game", "discipline_title", "event_title", "event_gender"]
events_us = df_us_dedup_medals[EVENT_KEY].drop_duplicates().reset_index(drop=True)
print("Unique USA events to scrape:", len(events_us))

Unique USA events to scrape: 704


## 4) Scraping helpers (Wikipedia + Wikidata + Wikitext)

### Date parsing rule
If Wikipedia provides a date range (e.g., `29 July – 1 August 2021`), we assign `event_date_end = 2021-08-01`.

### Methods
- **M1 (Infobox HTML):** fetch Wikipedia HTML via REST and extract infobox `Dates` row.
- **M2 (Wikidata):** map Wikipedia page → Wikidata QID → pull time claims `P585` (point in time), then `P582` (end), then `P580` (start).
- **M3 (Wikitext):** fetch page wikitext via MediaWiki API and extract `| dates = ...` / `| date = ...` from infobox template.


In [ ]:
def slug_to_year(slug_game: str) -> Optional[int]:
    m = re.search(r"(\d{4})", str(slug_game))
    return int(m.group(1)) if m else None


def wiki_search(query: str, limit: int = 8) -> List[Dict[str, Any]]:
    params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "srlimit": limit,
        "format": "json",
        "utf8": 1,
    }
    r = SESSION.get(WIKI_API, params=params, timeout=30)
    r.raise_for_status()
    return r.json().get("query", {}).get("search", [])


def choose_best_wiki_title(results: List[Dict[str, Any]], discipline: str, event: str, year: int) -> Optional[str]:
    if not results:
        return None
    target = f"{discipline} {event} {year} olympics".lower()
    scored = []
    for it in results:
        title = it.get("title", "")
        t = title.lower()
        score = 0
        if str(year) in t:
            score += 20
        if "olympic" in t:
            score += 15
        score += int(fuzz.token_set_ratio(t, target) * 0.5)
        scored.append((score, title))
    scored.sort(reverse=True, key=lambda x: x[0])
    return scored[0][1] if scored else None


def cache_read(path: Path) -> Optional[str]:
    if path.exists():
        return path.read_text(encoding="utf-8", errors="ignore")
    return None


def cache_write(path: Path, text: str) -> None:
    path.write_text(text, encoding="utf-8")


def wiki_html(title: str) -> str:
    safe = re.sub(r"[^a-zA-Z0-9._-]+", "_", title)[:200]
    cache_path = CACHE_DIR / f"html__{safe}.html"
    cached = cache_read(cache_path)
    if cached is not None:
        return cached

    url = f"{WIKI_REST}/{title.replace(' ', '_')}"
    r = SESSION.get(url, timeout=30)
    r.raise_for_status()
    cache_write(cache_path, r.text)
    return r.text


def wiki_wikitext(title: str) -> str:
    safe = re.sub(r"[^a-zA-Z0-9._-]+", "_", title)[:200]
    cache_path = CACHE_DIR / f"wikitext__{safe}.txt"
    cached = cache_read(cache_path)
    if cached is not None:
        return cached

    params = {
        "action": "query",
        "prop": "revisions",
        "rvprop": "content",
        "rvslots": "main",
        "format": "json",
        "titles": title,
        "redirects": 1,
    }
    r = SESSION.get(WIKI_API, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()
    pages = data.get("query", {}).get("pages", {})
    text = ""
    for _, page in pages.items():
        revs = page.get("revisions", [])
        if not revs:
            continue
        slots = revs[0].get("slots", {})
        text = slots.get("main", {}).get("*", "") or ""
    cache_write(cache_path, text)
    return text


def extract_infobox_dates_from_html(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "lxml")
    infobox = soup.find("table", class_=re.compile(r"infobox"))
    if not infobox:
        return None

    for row in infobox.find_all("tr"):
        th = row.find("th")
        td = row.find("td")
        if not th or not td:
            continue
        key = th.get_text(" ", strip=True).lower()
        if key in ("dates", "date"):
            return td.get_text(" ", strip=True)
    return None


def extract_infobox_dates_from_wikitext(wt: str) -> Optional[str]:
    # Look for common infobox params: | dates = ... or | date = ...
    # Stop at newline, pipe, or end of string.
    if not wt:
        return None

    # Prefer "dates" then "date"
    for key in ["dates", "date"]:
        m = re.search(rf"\|\s*{key}\s*=\s*([^\n\r\|]+)", wt, flags=re.IGNORECASE)
        if m:
            val = m.group(1).strip()
            # remove wiki markup like [[...]] and templates {{...}}
            val = re.sub(r"\[\[([^\]|]+)(\|[^\]]+)?\]\]", r"\1", val)
            val = re.sub(r"\{\{[^\}]+\}\}", "", val)
            val = re.sub(r"<[^>]+>", "", val).strip()
            return val if val else None
    return None


def parse_event_date_end(datestr: str, year: int) -> Optional[str]:
    if not datestr or not year:
        return None

    s = str(datestr).replace("\u2013", "-").replace("\u2014", "-")
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"\[\d+\]", "", s).strip()  # remove [1] refs

    if "-" in s:
        _, right = [x.strip() for x in s.split("-", 1)]
        for candidate in (right, f"{right} {year}"):
            try:
                end_dt = dtparser.parse(candidate, fuzzy=True, default=datetime(year, 1, 1))
                end_dt = end_dt.replace(year=year)
                return end_dt.date().isoformat()
            except Exception:
                pass
        return None

    for candidate in (s, f"{s} {year}"):
        try:
            dt = dtparser.parse(candidate, fuzzy=True, default=datetime(year, 1, 1))
            dt = dt.replace(year=year)
            return dt.date().isoformat()
        except Exception:
            pass
    return None


def wiki_title_to_wikidata_qid(title: str) -> Optional[str]:
    if not title:
        return None
    params = {
        "action": "query",
        "format": "json",
        "titles": title,
        "prop": "pageprops",
        "ppprop": "wikibase_item",
        "redirects": 1,
    }
    r = SESSION.get(WIKI_API, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()
    pages = data.get("query", {}).get("pages", {})
    for _, page in pages.items():
        qid = page.get("pageprops", {}).get("wikibase_item")
        if qid:
            return qid
    return None


def wikidata_get_claim_time_value(claim: Dict[str, Any]) -> Optional[str]:
    """
    Extract YYYY-MM-DD from a Wikidata time claim.
    Reject partial-precision dates (month/day == '00'), e.g. 1920-00-00.
    """
    try:
        time_str = claim["mainsnak"]["datavalue"]["value"]["time"]
        m = re.match(r"^[+-](\d{4})-(\d{2})-(\d{2})T", time_str)
        if not m:
            return None

        yyyy, mm, dd = m.group(1), m.group(2), m.group(3)

        # Reject partial precision (unknown month/day)
        if mm == "00" or dd == "00":
            return None

        return f"{yyyy}-{mm}-{dd}"
    except Exception:
        return None


def wikidata_get_event_date_end(qid: str, expected_year: int) -> Optional[str]:
    if not qid:
        return None

    params = {"action": "wbgetentities", "format": "json", "ids": qid, "props": "claims"}
    r = SESSION.get(WIKIDATA_API, params=params, timeout=30)
    r.raise_for_status()
    ent = r.json().get("entities", {}).get(qid, {})
    claims = ent.get("claims", {})

    for pid in ["P585", "P582", "P580"]:
        if pid in claims and claims[pid]:
            iso = wikidata_get_claim_time_value(claims[pid][0])
            if iso:
                year = int(iso[:4])
                if year == expected_year:
                    return iso   # only accept matching Olympic year

    return None

In [54]:
def scrape_event_date(slug_game: str, discipline: str, event: str, gender: str) -> Dict[str, Any]:
    """Multi-method date recovery. Returns dict with event_date_end and status."""
    year = slug_to_year(slug_game)
    if not year:
        return {"wiki_title": None, "raw_dates": None, "event_date_end": None, "status": "no_year_in_slug"}

    # --- Find a Wikipedia page title (once) ---
    queries = [
        f"\"{event}\" {discipline} {year} Olympics",
        f"{discipline} at the {year} Olympics {event}",
        f"\"{event}\" {year} Olympic {discipline}",
        f"{discipline} {event} {year} Olympic",
    ]

    wiki_title = None
    search_cache_key = re.sub(r"[^a-zA-Z0-9._-]+", "_", f"{discipline}__{event}__{year}")[:180]
    search_cache_path = CACHE_DIR / f"search__{search_cache_key}.json"
    if search_cache_path.exists():
        cached = pd.read_json(search_cache_path, typ="series").to_dict()
        wiki_title = cached.get("wiki_title")

    if not wiki_title:
        for q in queries:
            try:
                res = wiki_search(q, limit=10)
            except Exception:
                continue
            title = choose_best_wiki_title(res, discipline, event, year)
            if title:
                wiki_title = title
                break
        search_cache_path.write_text(pd.Series({"wiki_title": wiki_title}).to_json(), encoding="utf-8")

    if not wiki_title:
        return {"wiki_title": None, "raw_dates": None, "event_date_end": None, "status": "no_wiki_match"}

    # --- Method 1: Infobox Dates from HTML ---
    try:
        html = wiki_html(wiki_title)
        raw_dates = extract_infobox_dates_from_html(html)
        if raw_dates:
            iso = parse_event_date_end(raw_dates, year)
            if iso:
                return {"wiki_title": wiki_title, "raw_dates": raw_dates, "event_date_end": iso, "status": "ok_infobox"}
            # dates present but parse failed
            m1_out = {"wiki_title": wiki_title, "raw_dates": raw_dates, "event_date_end": None, "status": "date_parse_failed"}
        else:
            m1_out = {"wiki_title": wiki_title, "raw_dates": None, "event_date_end": None, "status": "no_infobox_dates"}
    except Exception as e:
        return {"wiki_title": wiki_title, "raw_dates": None, "event_date_end": None, "status": f"fetch_failed_html: {e}"}

    # --- Method 2: Wikidata fallback ---
    try:
        qid = wiki_title_to_wikidata_qid(wiki_title)
        if qid:
            wd = wikidata_get_event_date_end(qid, year)
            if wd:
                return {"wiki_title": wiki_title, "raw_dates": "Wikidata", "event_date_end": wd, "status": "ok_wikidata", "wikidata_qid": qid}
    except Exception as e:
        # keep going to method 3
        m1_out["wikidata_error"] = str(e)

    # --- Method 3: Wikitext infobox param fallback ---
    try:
        wt = wiki_wikitext(wiki_title)
        wt_dates = extract_infobox_dates_from_wikitext(wt)
        if wt_dates:
            iso = parse_event_date_end(wt_dates, year)
            if iso:
                return {"wiki_title": wiki_title, "raw_dates": wt_dates, "event_date_end": iso, "status": "ok_wikitext"}
            else:
                m1_out["wikitext_raw_dates"] = wt_dates
                m1_out["status"] = "date_parse_failed_wikitext"
        else:
            # preserve prior status, but note wikitext fail
            m1_out["wikitext_raw_dates"] = None
            if m1_out["status"] == "no_infobox_dates":
                m1_out["status"] = "no_dates_infobox_or_wikitext"
    except Exception as e:
        m1_out["wikitext_error"] = str(e)

    return m1_out

## 5) Run the scraper

This section:
- Runs a small sanity-check sample
- Performs a full scrape with incremental saving and resume support

In [ ]:
# Sanity-check on a small sample first
sample = events_us.sample(min(20, len(events_us)), random_state=42).reset_index(drop=True)

rows = []
for _, r in tqdm(sample.iterrows(), total=len(sample)):
    out = scrape_event_date(
        slug_game=r["slug_game"],
        discipline=r["discipline_title"],
        event=r["event_title"],
        gender=r["event_gender"],
    )
    rows.append({**r.to_dict(), **out})
    time.sleep(0.5)  # rate limit

sample_dates = pd.DataFrame(rows)
display(sample_dates.head(20))
print(sample_dates["status"].value_counts())

100%|██████████| 20/20 [00:14<00:00,  1.39it/s]


,slug_game,discipline_title,event_title,event_gender,wiki_title,raw_dates,event_date_end,status,wikitext_raw_dates
0,london-2012,Swimming,200m butterfly men,Men,Israel at the 2016 Summer Olympics,January 2017}},2012-01-01,ok_wikitext,NaN
1,tokyo-2020,Athletics,Men's 110m Hurdles,Men,Athletics at the 1996 Summer Olympics – Men's ...,28 & 29 July,2020-07-29,ok_infobox,NaN
2,tokyo-2020,Athletics,Women's 3000m Steeplechase,Women,3000 metres steeplechase,5 May 2017,2020-05-05,ok_wikitext,NaN
3,rio-2016,Volleyball,volleyball women,Women,Volleyball at the 2016 Summer Olympics,6–21 August 2016,2016-08-21,ok_infobox,NaN
4,london-2012,Athletics,4x400m relay men,Men,1949 Summer Deaflympics,2012,2012-01-01,ok_wikitext,NaN
5,london-2012,Swimming,200m individual medley women,Women,List of Norwegian records in swimming,April 2022}},2012-04-01,ok_wikitext,NaN
6,rio-2016,Swimming,200m breaststroke men,Men,List of deaf world records in swimming,25 July 2018,2016-07-25,ok_wikitext,NaN
7,london-2012,Athletics,high jump men,Men,Athletics at the 2012 Summer Olympics – Men's ...,5–7 August,2012-08-07,ok_infobox,NaN
8,london-2012,Swimming,50m freestyle men,Men,Swaziland at the 2008 Summer Olympics,10 May 1984,2012-05-10,ok_wikitext,NaN
9,tokyo-2020,Swimming,Women's 4 x 100m Medley Relay,Women,World and Olympic records set at the 2020 Summ...,August 2016}},2020-08-01,ok_wikitext,NaN


status
ok_wikitext                   12
ok_infobox                     7
date_parse_failed_wikitext     1
Name: count, dtype: int64


In [ ]:
# Full scrape (incremental + resumable)
OUT_PATH = Path("usa_event_dates_lookup.csv")

if OUT_PATH.exists():
    event_dates = pd.read_csv(OUT_PATH)
    print("Loaded existing lookup:", event_dates.shape)
else:
    event_dates = pd.DataFrame(columns=list(events_us.columns) + ["wiki_title","raw_dates","event_date_end","status","wikidata_qid","wikidata_error","wikitext_raw_dates","wikitext_error"])

done_keys = set(
    tuple(x) for x in event_dates[EVENT_KEY].values.tolist()
) if len(event_dates) else set()

to_do = events_us[~events_us.apply(lambda r: (r["slug_game"], r["discipline_title"], r["event_title"], r["event_gender"]) in done_keys, axis=1)]
print("Remaining USA events:", len(to_do))

rows = []
for _, r in tqdm(to_do.iterrows(), total=len(to_do)):
    out = scrape_event_date(
        slug_game=r["slug_game"],
        discipline=r["discipline_title"],
        event=r["event_title"],
        gender=r["event_gender"],
    )
    rows.append({**r.to_dict(), **out})

    time.sleep(0.5) 

    if len(rows) >= 100:
        event_dates = pd.concat([event_dates, pd.DataFrame(rows)], ignore_index=True)
        event_dates.to_csv(OUT_PATH, index=False)
        rows = []

if rows:
    event_dates = pd.concat([event_dates, pd.DataFrame(rows)], ignore_index=True)
    event_dates.to_csv(OUT_PATH, index=False)

print("Lookup table saved:", event_dates.shape)
display(event_dates.head())
print(event_dates["status"].value_counts())

Remaining USA events: 704


100%|██████████| 704/704 [08:34<00:00,  1.37it/s]

Lookup table saved: (704, 12)


,slug_game,discipline_title,event_title,event_gender,wiki_title,raw_dates,event_date_end,status,wikidata_qid,wikidata_error,wikitext_raw_dates,wikitext_error
0,beijing-2022,Freestyle Skiing,Men's Freeski Halfpipe,Men,New Zealand at the 2022 Winter Olympics,Wikidata,2022-01-01,ok_wikidata,Q48784914,NaN,NaN,NaN
1,beijing-2022,Freestyle Skiing,Men's Freeski Big Air,Men,New Zealand at the 2022 Winter Olympics,Wikidata,2022-01-01,ok_wikidata,Q48784914,NaN,NaN,NaN
2,beijing-2022,Freestyle Skiing,Women's Moguls,Women,Freestyle skiing at the 2022 Winter Olympics –...,"3, 6 February 2022",2022-02-06,ok_infobox,NaN,NaN,NaN,NaN
3,beijing-2022,Freestyle Skiing,Mixed Team Aerials,Mixed,Freestyle skiing at the 2022 Winter Olympics –...,10 February 2022,2022-02-10,ok_infobox,NaN,NaN,NaN,NaN
4,beijing-2022,Freestyle Skiing,Women's Aerials,Women,Freestyle skiing at the 2022 Winter Olympics –...,14 February 2022,2022-02-14,ok_infobox,NaN,NaN,NaN,NaN


status
ok_wikitext                                                                                                                                                                                   311
ok_infobox                                                                                                                                                                                    243
ok_wikidata                                                                                                                                                                                    73
date_parse_failed_wikitext                                                                                                                                                                     35
no_dates_infobox_or_wikitext                                                                                                                                                                   35
fetch_failed_html: expe

In [ ]:
# ---------------------------------------------------
# Drop events with missing event_date_end
# ---------------------------------------------------

# Check how many unique events are missing dates
missing_events = event_dates[event_dates["event_date_end"].isna()]
print("Unique events missing dates:", len(missing_events))

# Drop them from the lookup table
event_dates_clean = event_dates[event_dates["event_date_end"].notna()].copy()

print("Remaining unique events after drop:", len(event_dates_clean))
print("Dropped %:", round(100 * len(missing_events) / len(event_dates), 2), "%")

display(missing_events.head())

Unique events missing dates: 77
Remaining unique events after drop: 627
Dropped %: 10.94 %


,slug_game,discipline_title,event_title,event_gender,wiki_title,raw_dates,event_date_end,status,wikidata_qid,wikidata_error,wikitext_raw_dates,wikitext_error
31,tokyo-2020,Canoe Sprint,Women's Canoe Single 200m,Women,Canoeing at the 2020 Summer Olympics – Women's...,4 August 2021 (heats and quarterfinal) 5 Augus...,None,date_parse_failed_wikitext,NaN,NaN,4 August 2021 (heats and quarterfinal)5 August...,NaN
60,tokyo-2020,Artistic Gymnastics,Women's Balance Beam,Women,Gymnastics at the 2020 Summer Olympics – Women...,25 July 2021 (qualifying) 3 August 2021 (final),None,date_parse_failed_wikitext,NaN,NaN,25 July 2021 (qualifying)3 August 2021 (final),NaN
62,tokyo-2020,Artistic Gymnastics,Women's Team,Women,Gymnastics at the 2020 Summer Olympics – Women...,25 July (qualification) 27 July (final),None,date_parse_failed_wikitext,NaN,NaN,25 July (qualification) 27 July (final),NaN
63,tokyo-2020,Artistic Gymnastics,Women's Floor Exercise,Women,Gymnastics at the 2020 Summer Olympics – Women...,25 July 2021 (qualifying) 2 August 2021 (final),None,date_parse_failed_wikitext,NaN,NaN,25 July 2021 (qualifying)2 August 2021 (final),NaN
85,tokyo-2020,Athletics,Women's Shot Put,Women,Athletics at the 2020 Summer Olympics – Women'...,30 July 2021 (qualifying) 1 August 2021 (final),None,date_parse_failed_wikitext,NaN,NaN,30 July 2021 (qualifying)1 August 2021(final),NaN


## 6) Merge event dates back to the USA medal rows and export

We merge on: `slug_game, discipline_title, event_title, event_gender`.


In [58]:
# Merge lookup onto the USA dataset (row-level)
df_us_enriched = df_us.merge(
    event_dates_clean[EVENT_KEY + ["event_date_end","raw_dates","wiki_title","status","wikidata_qid"]],
    on=EVENT_KEY,
    how="left",
)

print("USA enriched shape:", df_us_enriched.shape)
display(df_us_enriched.head(10))

# Save enriched dataset
df_us_enriched.to_csv("usa_olympic_medals_with_event_dates.csv", index=False)
print("Wrote: usa_olympic_medals_with_event_dates.csv")

USA enriched shape: (870, 19)


,discipline_title,slug_game,event_title,event_gender,medal_type,participant_type,participant_title,athlete_url,athlete_full_name,country_name,country_code,country_3_letter_code,year,season,event_date_end,raw_dates,wiki_title,status,wikidata_qid
0,Freestyle Skiing,beijing-2022,Men's Freeski Halfpipe,Men,SILVER,Athlete,NaN,https://olympics.com/en/athletes/david-wise,David WISE,United States of America,US,USA,2022,W,2022-01-01,Wikidata,New Zealand at the 2022 Winter Olympics,ok_wikidata,Q48784914
1,Freestyle Skiing,beijing-2022,Men's Freeski Halfpipe,Men,BRONZE,Athlete,NaN,https://olympics.com/en/athletes/alex-ferreira,Alex FERREIRA,United States of America,US,USA,2022,W,2022-01-01,Wikidata,New Zealand at the 2022 Winter Olympics,ok_wikidata,Q48784914
2,Freestyle Skiing,beijing-2022,Men's Freeski Big Air,Men,SILVER,Athlete,NaN,https://olympics.com/en/athletes/colby-stevenson,Colby STEVENSON,United States of America,US,USA,2022,W,2022-01-01,Wikidata,New Zealand at the 2022 Winter Olympics,ok_wikidata,Q48784914
3,Freestyle Skiing,beijing-2022,Women's Moguls,Women,SILVER,Athlete,NaN,https://olympics.com/en/athletes/kauf,Jaelin KAUF,United States of America,US,USA,2022,W,2022-02-06,"3, 6 February 2022",Freestyle skiing at the 2022 Winter Olympics –...,ok_infobox,NaN
4,Freestyle Skiing,beijing-2022,Mixed Team Aerials,Mixed,GOLD,GameTeam,United States,NaN,NaN,United States of America,US,USA,2022,W,2022-02-10,10 February 2022,Freestyle skiing at the 2022 Winter Olympics –...,ok_infobox,NaN
5,Freestyle Skiing,beijing-2022,Women's Aerials,Women,BRONZE,Athlete,NaN,https://olympics.com/en/athletes/megan-nick,Megan NICK,United States of America,US,USA,2022,W,2022-02-14,14 February 2022,Freestyle skiing at the 2022 Winter Olympics –...,ok_infobox,NaN
6,Freestyle Skiing,beijing-2022,Men's Freeski Slopestyle,Men,SILVER,Athlete,NaN,https://olympics.com/en/athletes/nicholas-goepper,Nicholas GOEPPER,United States of America,US,USA,2022,W,2022-01-01,Wikidata,New Zealand at the 2022 Winter Olympics,ok_wikidata,Q48784914
7,Freestyle Skiing,beijing-2022,Men's Freeski Slopestyle,Men,GOLD,Athlete,NaN,https://olympics.com/en/athletes/alexander-hall-1,Alexander HALL,United States of America,US,USA,2022,W,2022-01-01,Wikidata,New Zealand at the 2022 Winter Olympics,ok_wikidata,Q48784914
8,Snowboard,beijing-2022,Women's Snowboard Halfpipe,Women,GOLD,Athlete,NaN,https://olympics.com/en/athletes/chloe-kim,Chloe KIM,United States of America,US,USA,2022,W,2022-02-03,3 February 2010,Australia at the 2010 Winter Olympics,ok_wikitext,NaN
9,Snowboard,beijing-2022,Mixed Team Snowboard Cross,Mixed,GOLD,GameTeam,United States of America 1,https://olympics.com/en/athletes/nick-baumgartner,Nick BAUMGARTNER,United States of America,US,USA,2022,W,2022-02-12,12 February,Snowboarding at the 2022 Winter Olympics – Mix...,ok_infobox,NaN


Wrote: usa_olympic_medals_with_event_dates.csv


## 7) Diagnostics

In [ ]:
coverage = (df_us_enriched["event_date_end"].notna().mean() * 100)
print(f"Event-date coverage (USA rows): {coverage:.1f}%")

missing_events = (
    df_us_enriched[df_us_enriched["event_date_end"].isna()][EVENT_KEY + ["status"]]
    .drop_duplicates()
    .sort_values(EVENT_KEY)
)
print("Missing unique USA events:", len(missing_events))
display(missing_events.head(50))

Event-date coverage (USA rows): 89.3%
Missing unique USA events: 77


,slug_game,discipline_title,event_title,event_gender,status
645,athens-2004,Athletics,100m men,Men,NaN
715,athens-2004,Boxing,75-81kg lightheavyweight men,Men,NaN
710,athens-2004,Gymnastics Artistic,horizontal bar men,Men,NaN
703,athens-2004,Gymnastics Artistic,individual all-round men,Men,NaN
704,athens-2004,Gymnastics Artistic,team competition women,Women,NaN
708,athens-2004,Gymnastics Artistic,uneven bars women,Women,NaN
692,athens-2004,Swimming,400m individual medley women,Women,NaN
689,athens-2004,Swimming,4x200m freestyle relay men,Men,NaN
693,athens-2004,Swimming,4x200m freestyle relay women,Women,NaN
659,athens-2004,Wrestling,"Light-Heavyweight, Freestyle (≤84 kilograms) Men",Men,NaN


In [ ]:
# Create daily USA medal counts + a weighted "performance score"

# Ensure datetime
df_us_enriched["event_date_end"] = pd.to_datetime(df_us_enriched["event_date_end"], errors="coerce")

df_medals = df_us_enriched.drop_duplicates(subset=MEDAL_KEY).copy()

valid_medals = {"GOLD", "SILVER", "BRONZE"}
df_medals = df_medals[df_medals["medal_type"].isin(valid_medals)].copy()

# Medal weights
MEDAL_WEIGHTS = {"GOLD": 3, "SILVER": 2, "BRONZE": 1}
df_medals["medal_points"] = df_medals["medal_type"].map(MEDAL_WEIGHTS).astype(int)

# Restrict to 2000–2024 (based on event date)
df_medals = df_medals[
    (df_medals["event_date_end"].dt.year >= 2000) &
    (df_medals["event_date_end"].dt.year <= 2024)
].copy()

# Daily aggregation
usa_medal_daily_all = (
    df_medals
    .groupby(["event_date_end", "season"])
    .agg(
        total_medals=("medal_type", "size"),
        gold=("medal_type", lambda s: (s == "GOLD").sum()),
        silver=("medal_type", lambda s: (s == "SILVER").sum()),
        bronze=("medal_type", lambda s: (s == "BRONZE").sum()),
        performance_score=("medal_points", "sum"),
    )
    .reset_index()
    .sort_values("event_date_end")
)

# Standard column order
cols = ["event_date_end", "total_medals", "gold", "silver", "bronze", "performance_score", "season"]
usa_medal_daily_all = usa_medal_daily_all[cols]

# Split into Summer/Winter
usa_medal_daily_summer = usa_medal_daily_all[usa_medal_daily_all["season"] == "S"].copy()
usa_medal_daily_winter = usa_medal_daily_all[usa_medal_daily_all["season"] == "W"].copy()

# Preview
print("ALL (S+W):", usa_medal_daily_all.shape)
print("SUMMER (S):", usa_medal_daily_summer.shape)
print("WINTER (W):", usa_medal_daily_winter.shape)

display(usa_medal_daily_all.head())
display(usa_medal_daily_summer.head())
display(usa_medal_daily_winter.head())

# Save 3 CSVs
usa_medal_daily_all.to_csv("usa_medal_daily_counts_and_score_ALL.csv", index=False)
usa_medal_daily_summer.to_csv("usa_medal_daily_counts_and_score_SUMMER.csv", index=False)
usa_medal_daily_winter.to_csv("usa_medal_daily_counts_and_score_WINTER.csv", index=False)

print("Wrote:")
print(" - usa_medal_daily_counts_and_score_ALL.csv")
print(" - usa_medal_daily_counts_and_score_SUMMER.csv")
print(" - usa_medal_daily_counts_and_score_WINTER.csv")

ALL (S+W): (281, 7)
SUMMER (S): (201, 7)
WINTER (W): (80, 7)


,event_date_end,total_medals,gold,silver,bronze,performance_score,season
0,2000-01-01,23,10,6,7,49,S
1,2000-02-01,13,3,5,5,24,S
2,2000-02-16,2,0,0,2,2,S
3,2000-03-01,1,0,0,1,1,S
4,2000-04-01,2,2,0,0,6,S


,event_date_end,total_medals,gold,silver,bronze,performance_score,season
0,2000-01-01,23,10,6,7,49,S
1,2000-02-01,13,3,5,5,24,S
2,2000-02-16,2,0,0,2,2,S
3,2000-03-01,1,0,0,1,1,S
4,2000-04-01,2,2,0,0,6,S


,event_date_end,total_medals,gold,silver,bronze,performance_score,season
26,2002-02-09,1,0,1,0,2,W
27,2002-02-11,2,0,1,1,3,W
28,2002-02-12,4,1,2,1,8,W
29,2002-02-15,1,0,0,1,1,W
30,2002-02-19,1,1,0,0,3,W


Wrote:
 - usa_medal_daily_counts_and_score_ALL.csv
 - usa_medal_daily_counts_and_score_SUMMER.csv
 - usa_medal_daily_counts_and_score_WINTER.csv
